# UNCP 3 | Model and executed results

کد مدل و یک بررسی اجرایی در همین دفترچه اجرا شده‌اند. اعداد سه seed از فایل‌های نتیجهٔ آزمایش کامل که پیش‌تر اجرا شده‌اند، در سلول آخر ثبت و با میانگین همان فایل‌ها کنترل می‌شوند. این دفترچه آموزش کامل سه seed را دوباره اجرا نمی‌کند؛ برای بازآموزی، داده، کد benchmark و تنظیمات همان پروژه لازم است.

In [1]:
import torch
from torch import nn
import torch.nn.functional as F
import math
import torch

def sinusoidal_positions(max_context, dimension):
    pe = torch.zeros(max_context, dimension)
    positions = torch.arange(max_context)[:, None]
    div = torch.exp(torch.arange(0, dimension, 2) * (-math.log(10000.0) / dimension))
    pe[:, 0::2] = torch.sin(positions * div)
    pe[:, 1::2] = torch.cos(positions * div)
    return pe

"""UNCP 3: GRU, Transformer, and a PyTorch SSD (Mamba-2-style) branch.

The SSD branch uses input-dependent scalar decay and B/C vectors per head.
It implements the recurrent form of structured state-space duality in plain
PyTorch, without claiming use of the optimized official CUDA Mamba2 kernel.
"""
import torch
from torch import nn
import torch.nn.functional as F



class SSDLayer(nn.Module):
    def __init__(self, dim, heads=4, state_size=16):
        super().__init__()
        assert dim % heads == 0
        self.heads, self.width, self.state_size = heads, dim // heads, state_size
        self.in_proj = nn.Linear(dim, dim + 2 * heads * state_size + heads + dim)
        self.out_proj = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        batch, steps, dim = x.shape
        z = self.in_proj(x)
        w = self.heads * self.state_size
        u, b, c, decay, gate = torch.split(z, [dim, w, w, self.heads, dim], dim=-1)
        u = u.reshape(batch, steps, self.heads, self.width)
        b = b.reshape(batch, steps, self.heads, self.state_size)
        c = c.reshape(batch, steps, self.heads, self.state_size)
        decay = torch.sigmoid(decay)
        state = x.new_zeros(batch, self.heads, self.width, self.state_size)
        outputs = []
        for t in range(steps):
            state = decay[:, t, :, None, None] * state + u[:, t, :, :, None] * b[:, t, :, None, :]
            outputs.append((state * c[:, t, :, None, :]).sum(-1))
        y = torch.stack(outputs, dim=1).reshape(batch, steps, dim)
        return self.norm(x + self.out_proj(y * F.silu(gate)))


class UNCP3NextClick(nn.Module):
    def __init__(self, vocab_size, n_classes, config):
        super().__init__()
        dim, hidden = config['embedding_dim'], config['hidden_dim']
        self.embedding = nn.Embedding(vocab_size, dim, padding_idx=0)
        self.gru = nn.GRU(dim, hidden, batch_first=True)
        layer = nn.TransformerEncoderLayer(dim, config['attention_heads'],
            config['transformer_feedforward'], config['dropout'], batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, config['transformer_layers'], enable_nested_tensor=False)
        self.ssd = SSDLayer(dim, config['attention_heads'])
        self.register_buffer('positional', sinusoidal_positions(config['max_context'], dim))
        self.transform_projection = nn.Linear(dim, hidden)
        self.ssd_projection = nn.Linear(dim, hidden)
        self.gate = nn.Linear(3 * hidden, 3)
        self.norm = nn.LayerNorm(hidden)
        self.dropout = nn.Dropout(config['dropout'])
        self.output = nn.Linear(hidden, n_classes)

    def forward(self, x, lengths):
        x = x[:, :int(lengths.max())]
        batch = torch.arange(len(x), device=x.device)
        embedded = self.embedding(x)
        gru, _ = self.gru(embedded)
        gru = gru[batch, lengths - 1]
        transformer = self.transformer(embedded + self.positional[:x.shape[1]], src_key_padding_mask=x.eq(0))
        transformer = self.transform_projection(transformer[batch, lengths - 1])
        ssd = self.ssd_projection(self.ssd(embedded)[batch, lengths - 1])
        states = torch.stack([gru, transformer, ssd], dim=1)
        weights = torch.softmax(self.gate(torch.cat([gru, transformer, ssd], dim=-1)), dim=-1)
        fused = self.norm((states * weights.unsqueeze(-1)).sum(dim=1))
        return self.output(self.dropout(fused))


In [2]:
cfg=dict(embedding_dim=32,hidden_dim=64,attention_heads=4,transformer_feedforward=64,dropout=.2,transformer_layers=2,max_context=32)
model=UNCP3NextClick(10,5,cfg)
x=torch.tensor([[2,2,3,0],[4,5,6,7]])
lengths=torch.tensor([3,4])
out=model(x,lengths)
assert out.shape==(2,5) and torch.isfinite(out).all()
print('UNCP 3 forward pass OK:',tuple(out.shape))

UNCP 3 forward pass OK: (2, 5)


## Recorded experiment results

The following cell executes a consistency check over the recorded three-seed result rows and displays them. It does not retrain the model.

In [3]:
import json, numpy as np
record = json.loads("{\"summary\": {\"protocol\": \"grouped_user\", \"model\": \"uncp3\", \"seeds\": \"3\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.6112188067325147\", \"top3\": \"0.8139942767469768\", \"macro_target_top1\": \"0.5079396235824037\", \"macro_target_top3\": \"0.6855106314109176\", \"macro_user_top1\": \"0.6255331114932504\", \"macro_user_top3\": \"0.8202086292559252\", \"cluster_ci95_top1_low\": \"0.5595705902800073\", \"cluster_ci95_top1_high\": \"0.6622155941606191\", \"cluster_ci95_top3_low\": \"0.7800400898603109\", \"cluster_ci95_top3_high\": \"0.8387978780894642\", \"top1_seed_sd\": \"0.022853957728493367\", \"top3_seed_sd\": \"0.008237389744573671\", \"macro_target_top1_seed_sd\": \"0.03910456104403911\", \"macro_target_top3_seed_sd\": \"0.025677621025289022\", \"macro_user_top1_seed_sd\": \"0.025671229233503633\", \"macro_user_top3_seed_sd\": \"0.008054975730631787\"}, \"runs\": [{\"protocol\": \"grouped_user\", \"model\": \"uncp3\", \"seed\": \"11\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.5947567617465153\", \"top3\": \"0.809009507984861\", \"macro_target_top1\": \"0.47728998319684945\", \"macro_target_top3\": \"0.6658837215105331\", \"macro_user_top1\": \"0.6036999829162267\", \"macro_user_top3\": \"0.812148212883942\", \"best_epoch\": \"19\", \"epochs_run\": \"24\"}, {\"protocol\": \"grouped_user\", \"model\": \"uncp3\", \"seed\": \"23\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.6373119172897628\", \"top3\": \"0.8235022616080495\", \"macro_target_top1\": \"0.5519805777876855\", \"macro_target_top3\": \"0.7145706062371812\", \"macro_user_top1\": \"0.6538133701687139\", \"macro_user_top3\": \"0.8282581532983488\", \"best_epoch\": \"30\", \"epochs_run\": \"30\"}, {\"protocol\": \"grouped_user\", \"model\": \"uncp3\", \"seed\": \"37\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.6015877411612665\", \"top3\": \"0.80947106064802\", \"macro_target_top1\": \"0.4945483097626758\", \"macro_target_top3\": \"0.6760775664850387\", \"macro_user_top1\": \"0.6190859813948106\", \"macro_user_top3\": \"0.8202195215854851\", \"best_epoch\": \"20\", \"epochs_run\": \"25\"}], \"behavior\": null}")
summary, runs = record['summary'], record['runs']
top1 = float(summary.get('test_mean_top1', summary.get('top1')))
top3 = float(summary.get('test_mean_top3', summary.get('top3')))
seed_top1 = np.array([float(row['top1']) for row in runs])
seed_top3 = np.array([float(row['top3']) for row in runs])
assert np.isclose(seed_top1.mean(), top1, atol=1e-9)
assert np.isclose(seed_top3.mean(), top3, atol=1e-9)
print('Recorded grouped-user experiment: 10,833 test targets, 11 held-out users')
print(f'UNCP 3  Top-1: {100*top1:.2f}%  Top-3: {100*top3:.2f}%')
print('Seeds 11, 23, 37 Top-3:', [round(100*x,2) for x in seed_top3])
if record['behavior']:
    for group in record['behavior']:
        print(group['group'], 'n=',group['n'],'UNCP6 Top-3=',round(100*group['uncp6_top3'],2),'UNCP5 Top-3=',round(100*group['uncp5_top3'],2))

Recorded grouped-user experiment: 10,833 test targets, 11 held-out users
UNCP 3  Top-1: 61.12%  Top-3: 81.40%
Seeds 11, 23, 37 Top-3: [np.float64(80.9), np.float64(82.35), np.float64(80.95)]
